## Grader Validation
This notebook contains a basic elaboration of the grader validation data.

In [ ]:
import pandas as pd
from datasets import load_dataset

In [ ]:
def compute_metrics(human_answers: list[int], model_answers: list[int]) -> dict[str, float]:
	"""
	Compute accuracy, precision, recall, and F1 score for the model's answers compared to human answers.
	Consider human answers as the ground truth and model answers as the predictions.
	"""
	assert len(human_answers) == len(model_answers), "The length of human answers and model answers must be the same."

	tp = sum(1 for h, m in zip(human_answers, model_answers) if h == 1 and m == 1)
	fp = sum(1 for h, m in zip(human_answers, model_answers) if h == 0 and m == 1)
	fn = sum(1 for h, m in zip(human_answers, model_answers) if h == 1 and m == 0)
	tn = sum(1 for h, m in zip(human_answers, model_answers) if h == 0 and m == 0)	
	accuracy = (tp + tn) / len(human_answers) if len(human_answers) > 0 else 0
	precision = tp / (tp + fp) if (tp + fp) > 0 else 0
	recall = tp / (tp + fn) if (tp + fn) > 0 else 0
	f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
	fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
	return {
		"accuracy": accuracy,
		"precision": precision,
		"recall": recall,
		"fpr": fpr,
		"f1_score": f1_score
	}

In [ ]:
df_text = pd.read_csv("../data/validation/text_output_valid.csv")
df_img = pd.read_csv("../data/validation/img_output_valid.csv")
df_img_2 = pd.read_csv("../data/validation/img_output_valid_2.csv")
df_img = pd.concat([df_img, df_img_2], ignore_index=True)

valid_ids = load_dataset("blind-spots-neurips/blind-spots-bench", split="test").to_pandas()["QID"]

print(f"Filtering out: {set(df_img['id']) - set(valid_ids)} from image data")
print(f"Filtering out: {set(df_text['id']) - set(valid_ids)} from text data")

df_img = df_img[df_img["id"].isin(valid_ids)]
df_text = df_text[df_text["id"].isin(valid_ids)]

df_text = df_text[df_text["human_and_grader_scores_match"].notna()]

df_img.rename(columns={"verifier_score": "grader_score", "model": "model_name"}, inplace=True)
df_img["grader_score"] = df_img["grader_score"].apply(lambda x: 1 if x == "C" else 0)
df_img["human_score"]= df_img["human_score"].apply(lambda x: 1 if x == "C" else 0)

df_text["grader_score"] = df_text["grader_score"].apply(lambda x: 1 if x else 0)
df_text["human_score"] = df_text["human_score"].apply(lambda x: 1 if x else 0)


### General agreement check

In [ ]:

for df, modality in zip([df_text, df_img], ["text-outputs", "image-outputs"]):
	print(modality)
	print(f"\tNumber of samples: {len(df)}")
	print("\tDistribution of grader scores: {} correct, {} incorrect".format(df["grader_score"].sum(), len(df) - df["grader_score"].sum()))
	print("\tDistribution of human scores: {} correct, {} incorrect".format(df["human_score"].sum(), len(df) - df["human_score"].sum()))
	metrics = compute_metrics(df["human_score"].tolist(), df["grader_score"].tolist())
	print("\tMetrics:")
	for metric_name, metric_value in metrics.items():
		print(f"\t\t{metric_name}: {metric_value:.4f}")

### Pro-Google bias in grading

In [ ]:
for df, modality in zip([df_text, df_img], ["text-outputs", "image-outputs"]):
	print(modality)

	for subset in ["google", "others"]:		
		print(f"\tSubset: {subset}")
		if subset == "google":
			tmp_df = df[(df["model_name"].str.contains("gemini", case=False)) | df["model_name"].str.contains("gemma", case=False)].copy()
		else:
			tmp_df = df[~(df["model_name"].str.contains("gemini", case=False)) & ~df["model_name"].str.contains("gemma", case=False)].copy()
	
		print("\t\tDistribution of grader scores: {} correct, {} incorrect".format(tmp_df["grader_score"].sum(), len(tmp_df) - tmp_df["grader_score"].sum()))
		print("\t\tDistribution of human scores: {} correct, {} incorrect".format(tmp_df["human_score"].sum(), len(tmp_df) - tmp_df["human_score"].sum()))

		print(f"\t\tNumber of samples: {len(tmp_df)}")
		metrics = compute_metrics(tmp_df["human_score"].tolist(), tmp_df["grader_score"].tolist())
		print("\t\tMetrics:")
		for metric_name, metric_value in metrics.items():
			print(f"\t\t\t{metric_name}: {metric_value:.4f}")

### Export LaTeX tables

In [ ]:
from pathlib import Path

ASSETS_DIR = Path("../assets")
ASSETS_DIR.mkdir(exist_ok=True)


def fmt_metric(value: float) -> str:
	return f"{value:.3f}"


def summarize_validation_row(df: pd.DataFrame, output_type: str) -> dict[str, int | float | str]:
	metrics = compute_metrics(df["human_score"].tolist(), df["grader_score"].tolist())
	grader_correct = int(df["grader_score"].sum())
	human_correct = int(df["human_score"].sum())
	return {
		"output_type": output_type,
		"n": len(df),
		"grader_correct": grader_correct,
		"grader_incorrect": len(df) - grader_correct,
		"human_correct": human_correct,
		"human_incorrect": len(df) - human_correct,
		**metrics,
	}


def google_solver_mask(df: pd.DataFrame) -> pd.Series:
	return df["model_name"].str.contains("gemini", case=False, na=False) | df["model_name"].str.contains("gemma", case=False, na=False)


def summarize_bias_rows(df: pd.DataFrame, output_type: str) -> list[dict[str, int | float | str]]:
	rows = []
	for subset_name, subset_df in [
		("Google", df[google_solver_mask(df)]),
		("Others", df[~google_solver_mask(df)]),
	]:
		metrics = compute_metrics(subset_df["human_score"].tolist(), subset_df["grader_score"].tolist())
		rows.append(
			{
				"output_type": output_type,
				"solver_subset": subset_name,
				"n": len(subset_df),
				"human_correct": int(subset_df["human_score"].sum()),
				"grader_correct": int(subset_df["grader_score"].sum()),
				**metrics,
			}
		)
	return rows


def build_pipeline_validation_table(rows: list[dict[str, int | float | str]]) -> str:
	bs = chr(92)
	newline = " " + bs * 2
	lines = [
		f"{bs}begin{{table}}[t]",
		f"{bs}caption{{Pipeline Validation Results: agreement scores between human-annotated correctness scores and values obtained with the automatic evaluation pipeline.}}",
		f"{bs}label{{tab:pipeline_validation}}",
		f"{bs}centering",
		f"{bs}small",
		f"{bs}begin{{tabular}}{{lccccccccc}}",
		f"{bs}toprule",
		f"Output type & $n$ & {bs}multicolumn{{2}}{{c}}{{Grader}} & {bs}multicolumn{{2}}{{c}}{{Human}} & {bs}textbf{{Acc.}} & {bs}textbf{{Prec.}} & {bs}textbf{{Rec.}} & {bs}textbf{{F1}}" + newline,
		f"{bs}cmidrule(lr){{3-4}}{bs}cmidrule(lr){{5-6}}",
		" & & Correct & Incorrect & Correct & Incorrect & & & & " + newline,
		f"{bs}midrule",
	]
	for row in rows:
		lines.append(
			" & ".join(
				[
					str(row["output_type"]),
					str(row["n"]),
					str(row["grader_correct"]),
					str(row["grader_incorrect"]),
					str(row["human_correct"]),
					str(row["human_incorrect"]),
					fmt_metric(row["accuracy"]),
					fmt_metric(row["precision"]),
					fmt_metric(row["recall"]),
					fmt_metric(row["f1_score"]),
				]
			)
			+ newline
		)
	lines.extend([f"{bs}bottomrule", f"{bs}end{{tabular}}", f"{bs}end{{table}}"])
	return "\n".join(lines) + "\n"


def build_pipeline_validation_bias_table(rows: list[dict[str, int | float | str]]) -> str:
	bs = chr(92)
	newline = " " + bs * 2
	lines = [
		f"{bs}begin{{table}}[t]",
		f"{bs}centering",
		f"{bs}caption{{",
		"Disaggregated grader validation by solver model family. We report agreement between the automatic grader and human labels, together with false positive rate (FPR) and recall. FPR measures the rate at which the grader incorrectly marks a human-incorrect response as correct, and is therefore the most relevant metric for detecting potential overestimation of a model family's performance.",
		f"}}",
		f"{bs}label{{tab:pipeline_validation_bias}}",
		f"{bs}small",
		f"{bs}begin{{tabular}}{{llrrrrrr}}",
		f"{bs}toprule",
		f"Output type & Solver subset & $n$ & Human correct & Grader correct & {bs}textbf{{Accuracy}} & {bs}textbf{{FPR}} & {bs}textbf{{Recall}}" + newline,
		f"{bs}midrule",
	]
	previous_output_type = None
	for row in rows:
		if previous_output_type is not None and row["output_type"] != previous_output_type:
			lines.append(f"{bs}midrule")
		output_type = row["output_type"] if row["output_type"] != previous_output_type else ""
		lines.append(
			" & ".join(
				[
					str(output_type),
					str(row["solver_subset"]),
					str(row["n"]),
					str(row["human_correct"]),
					str(row["grader_correct"]),
					fmt_metric(row["accuracy"]),
					fmt_metric(row["fpr"]),
					fmt_metric(row["recall"]),
				]
			)
			+ newline
		)
		previous_output_type = row["output_type"]
	lines.extend([f"{bs}bottomrule", f"{bs}end{{tabular}}", f"{bs}end{{table}}"])
	return "\n".join(lines) + "\n"


validation_rows = [
	summarize_validation_row(df_text, "Text"),
	summarize_validation_row(df_img, "Image"),
]
bias_rows = summarize_bias_rows(df_text, "Text") + summarize_bias_rows(df_img, "Image")

pipeline_validation_table = build_pipeline_validation_table(validation_rows)
pipeline_validation_bias_table = build_pipeline_validation_bias_table(bias_rows)

(ASSETS_DIR / "notebooks/pipeline_validation.tex").write_text(pipeline_validation_table, encoding="utf-8")
(ASSETS_DIR / "notebooks/pipeline_validation_bias.tex").write_text(pipeline_validation_bias_table, encoding="utf-8")

print(f"Wrote {(ASSETS_DIR / 'notebooks/pipeline_validation.tex').resolve()}")
print(f"Wrote {(ASSETS_DIR / 'notebooks/pipeline_validation_bias.tex').resolve()}")